# ⚡ Swizzle Enterprise Worker — Colab Pro GPU Inference Node (NVIDIA A100 / L4)
This notebook runs **vLLM** with **distributed prefix caching** and **N-Gram Prompt Lookup Speculative Decoding (PLD)** on Colab Pro's A100/L4 GPU, exposing a secure HTTPS endpoint via **Cloudflare Zero Trust Named Tunnels (`api.withswizzle.com`)** or instant TryCloudflare tunnels.

In [ ]:
# 1. Verify GPU Allocation (Expect NVIDIA A100-SXM4-40GB/80GB or L4)
!nvidia-smi

In [ ]:
# 2. Install vLLM and Cloudflared Tunnel Binary
!pip install -q vllm
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ vLLM and cloudflared installed successfully!")

In [ ]:
# 3. Start vLLM Model Server with Prefix Caching & N-Gram Speculative Decoding
import subprocess, time, socket, os, sys

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"  #@param {type:"string"}
MAX_MODEL_LEN = "8192"  #@param ["4096", "8192", "16384", "32768"]
GPU_MEM_UTIL = "0.90"   #@param ["0.85", "0.90", "0.92", "0.95"]
ENABLE_SPECULATIVE_PLD = False  #@param {type:"boolean"}

print(f"🚀 Spawning vLLM worker on port 8000 with model {MODEL_ID} (Context: {MAX_MODEL_LEN}, GPU Util: {GPU_MEM_UTIL})...")
vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--enable-prefix-caching",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--trust-remote-code",
    "--port", "8000"
]

if ENABLE_SPECULATIVE_PLD:
    # Newer vLLM CLI spec flags
    vllm_cmd.extend([
        "--spec-method", "ngram",
        "--spec-tokens", "5"
    ])

# Pipe stdout & stderr to vllm.log so errors are immediately visible
vllm_log = open("vllm.log", "w", buffering=1)
vllm_process = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)

# Wait up to 360 seconds (6 minutes) for model download + CUDA compilation
ready = False
print("⏳ Downloading weights & compiling CUDA graphs (this takes 2-4 minutes)...")
for attempt in range(180):
    # 1. Check if process crashed
    ret = vllm_process.poll()
    if ret is not None:
        vllm_log.flush()
        print(f"\n❌ vLLM process exited unexpectedly with return code {ret}!")
        print("="*50 + " vllm.log tail " + "="*50)
        with open("vllm.log", "r") as f:
            lines = f.readlines()
            print("".join(lines[-40:]))
        raise RuntimeError(f"vLLM failed to start. See logs above.")

    # 2. Check if port is open
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1)
        s.connect(("127.0.0.1", 8000))
        s.close()
        ready = True
        print("\n\n✅ vLLM OpenAI API Server is READY on port 8000 (Prefix Caching + PLD Active)!")
        break
    except Exception:
        time.sleep(2)
        if attempt % 5 == 0:
            print(".", end="", flush=True)

if not ready:
    print("\n⚠️ Timed out waiting for port 8000.")
    print("="*50 + " vllm.log tail " + "="*50)
    with open("vllm.log", "r") as f:
        lines = f.readlines()
        print("".join(lines[-40:]))
    raise TimeoutError("vLLM server startup timed out after 6 minutes.")

In [ ]:
# 4. Launch Cloudflare Tunnel (Named Tunnel api.withswizzle.com or Ephemeral TryCloudflare)
import re, time, urllib.request, subprocess
from IPython.display import display, HTML

# Optional: Paste your Cloudflare Zero Trust Tunnel Token to serve permanently on api.withswizzle.com
# Leave blank for an instant temporary https://*.trycloudflare.com URL
CLOUDFLARE_TUNNEL_TOKEN = ""  #@param {type:"string"}

# Anti-sleep background audio heartbeat (prevents Chrome tab throttling)
display(HTML('''
    <audio id="silent-heartbeat" loop autoplay>
        <source src="data:audio/wav;base64,UklGRiQAAABXQVZFZm10IBAAAAABAAEAQB8AAEAfAAABAAgAZGF0YQAAAAA=" type="audio/wav">
    </audio>
    <script>
        console.log("⚡ Swizzle Anti-Sleep Heartbeat active.");
    </script>
'''))

def start_tunnel():
    if CLOUDFLARE_TUNNEL_TOKEN and len(CLOUDFLARE_TUNNEL_TOKEN.strip()) > 10:
        print("🔗 Connecting to Cloudflare Named Tunnel (api.withswizzle.com)...")
        tunnel_cmd = ["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", CLOUDFLARE_TUNNEL_TOKEN.strip()]
        p = subprocess.Popen(tunnel_cmd, stderr=subprocess.PIPE, universal_newlines=True)
        public_url = "https://api.withswizzle.com"
        print("\n" + "="*70)
        print(f"⚡ SWIZZLE NAMED TUNNEL LIVE: {public_url}")
        print("="*70)
    else:
        print("🔗 Spawning ephemeral quick tunnel (*.trycloudflare.com)...")
        tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://localhost:8000"]
        p = subprocess.Popen(tunnel_cmd, stderr=subprocess.PIPE, universal_newlines=True)
        public_url = None
        for line in p.stderr:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                public_url = match.group(0)
                print("\n" + "="*70)
                print(f"⚡ SWIZZLE UPSTREAM WORKER LIVE URL: {public_url}")
                print("="*70)
                break
    return p, public_url

tunnel_process, public_url = start_tunnel()

# Watchdog loop: keeps vLLM warm and restarts tunnel if dropped
uptime_ticks = 0
while True:
    time.sleep(30)
    uptime_ticks += 1
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5)
        if uptime_ticks % 10 == 0:
            print(f"[Watchdog] 🟢 Worker healthy | Uptime: {uptime_ticks*30}s | Tunnel: {public_url}")
    except Exception as e:
        print(f"[Watchdog] ⚠️ Ping warning: {e}")

    if tunnel_process.poll() is not None:
        print("[Watchdog] 🔄 Cloudflare tunnel dropped. Auto-recovering...")
        tunnel_process, public_url = start_tunnel()
